In [ ]:
import numpy as np
import math
from scipy.constants import m_e, c, e, hbar, physical_constants, epsilon_0
import tqdm
import numba
import matplotlib.pyplot as plt
from ionization_routines import get_fraction_and_temperature_multispecies, \
    process_intensity_array_multispecies, load_intensity_profile, \
    get_adk_parameters

# Temperature from ionization

In [ ]:
# Compute ADK parameters
species_keys, adk_prefactors, adk_powers, adk_exp_prefactors, source_indices, target_indices, charges = get_adk_parameters()

In [ ]:
a0 = 0.05
tau = 30e-15 # FWHM duration
lambd = 0.8e-6
ell = np.array([0,1])

default_populations = np.array([0.95, 0.0, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0])

final_pops, T, _ = get_fraction_and_temperature_multispecies(a0, tau, lambd, ell,
                                                            adk_prefactors, adk_powers, adk_exp_prefactors,
                                                            source_indices, target_indices, charges,
                                                            default_populations)
print(f"Final populations: {final_pops}")
print(f"Temperature: {T:.2f} eV")


In [ ]:
tau_arr = 10**np.linspace(-14, -11.5, 40)
lambd = 0.8e-6

# Define initial populations for multi-species calculation
# 95% H, 5% N (all neutral)
initial_pops = np.array([0.95, 0.0, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0])

# Linear polarization, a0=0.05
a0 = 0.05
ell = np.array([0,1])
T_arr = [ get_fraction_and_temperature_multispecies( a0, tau, lambd, ell,
                                                    adk_prefactors, adk_powers, adk_exp_prefactors,
                                                    source_indices, target_indices, charges,
                                                    initial_pops )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'b-', label='Linear polarization, a0=0.05' )

# Linear polarization, a0=0.025
a0 = 0.025
ell = np.array([0,1])
T_arr = [ get_fraction_and_temperature_multispecies( a0, tau, lambd, ell,
                                                    adk_prefactors, adk_powers, adk_exp_prefactors,
                                                    source_indices, target_indices, charges,
                                                    initial_pops )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'b--', label='Linear polarization, a0=0.025' )

# Circular polarization, a0=0.05
a0 = 0.05
ell = np.array([1,1])/2**.5
T_arr = [ get_fraction_and_temperature_multispecies( a0, tau, lambd, ell,
                                                    adk_prefactors, adk_powers, adk_exp_prefactors,
                                                    source_indices, target_indices, charges,
                                                    initial_pops )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'r-', label='Circular polarization, a0=0.05' )

# Circular polarization, a0=0.025
a0 = 0.025
ell = np.array([1,1])/2**.5
T_arr = [ get_fraction_and_temperature_multispecies( a0, tau, lambd, ell,
                                                    adk_prefactors, adk_powers, adk_exp_prefactors,
                                                    source_indices, target_indices, charges,
                                                    initial_pops )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'r--', label='Circular polarization, a0=0.025' )

plt.grid()
plt.legend(loc=0)
plt.xlabel('Laser duration (s)')
plt.ylabel('Temperature (eV)')
plt.title('Temperature vs Pulse Duration\n(H + 5% N dopant)')
plt.show()


In [ ]:
# Parameters
lambd = 0.8e-6
tau = 50e-15
ell = np.array([1, 0])  # linear polarization

# Create 1D radial intensity profile
r_1d = np.linspace(0, 100e-6, 100)
w0_1d = 20e-6  # beam waist
I0_1d = 1e19  # peak intensity in W/m^2
intensity_1d = I0_1d * np.exp(-2*r_1d**2/w0_1d**2)


# Create sample file for demonstration
np.savetxt('sample_intensity_hn.txt', intensity_1d)

# Load intensity profile
intensity_loaded = load_intensity_profile('sample_intensity_hn.txt')

# Compare different dopant concentrations
concentrations = [
    (np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), "Pure H", "blue"),
    (np.array([0.95, 0.0, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0]), "95% H + 5% N", "red"),
    (np.array([0.9, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0]), "90% H + 10% N", "green")
]

plt.figure(figsize=(12, 8))


plt.subplot(221)
for pops, label, color in concentrations:
    all_pops, T_eV = process_intensity_array_multispecies(intensity_loaded, lambd, tau, ell,
        adk_prefactors, adk_powers, adk_exp_prefactors, source_indices, target_indices, charges, species_keys, pops)
    plt.plot(r_1d*1e6, T_eV, color=color, linewidth=2, label=label)

plt.xlabel('Radius (μm)')
plt.ylabel('Temperature (eV)')
plt.title('Temperature Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot temperature in log scale
plt.subplot(223)
for pops, label, color in concentrations:
    all_pops, T_eV = process_intensity_array_multispecies(intensity_loaded, lambd, tau, ell,
        adk_prefactors, adk_powers, adk_exp_prefactors, source_indices, target_indices, charges, species_keys, pops)
    plt.plot(r_1d*1e6, T_eV, color=color, linewidth=2, label=label)

plt.xlabel('Radius (μm)')
plt.ylabel('Temperature (ev)')
plt.title('Temperature Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot intensity profile for reference
plt.subplot(224)
plt.plot(r_1d*1e6, intensity_loaded/1000 * 1e-15, 'k-', linewidth=2)
plt.xlabel('Radius (μm)')
plt.ylabel('Intensity (10^15 W/cm²)')
plt.title('Input Intensity Profile')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Clean up
import os
os.remove('sample_intensity_hn.txt')


# Ionization volume for a focusing pulse

In [ ]:
a0 = 0.05
w0 = 5.e-6
lambd = 0.8e-6
tau = 1.5e-12 # FWHM duration
ell = np.array([0,1])

zr = np.pi*w0**2/lambd

In [ ]:
zmax = 400e-6
rmax = 50e-6
r, z = np.meshgrid( np.linspace(-rmax, rmax, 50), np.linspace(-zmax, zmax, 50), indexing='ij' )
a = a0/np.sqrt(1 + z**2/zr**2)*np.exp( -r**2/w0**2/(1+z**2/zr**2) )
intensity = 2 * epsilon_0 * c**3 * (np.pi * m_e * c * a / (e * lambd))**2

# Compute ionization over a 2D (r,z) grid
pops, T = process_intensity_array_multispecies(
    intensity,
    lambd, tau, ell,
    adk_prefactors, adk_powers, adk_exp_prefactors,
    source_indices, target_indices, charges, species_keys,
    initial_pops, output_file='plasma_from_file_2d.h5',
    grid_extent={'r': [r.min(), r.max()], 'z': [z.min(), z.max()]})


# Compute electron density
n = (pops * charges).sum(axis=-1)

In [ ]:
plt.figure(figsize=(5,8))

extent = 1.e6*np.array([-zmax, zmax, -rmax, rmax])

plt.subplot(311)
plt.imshow(a, extent=extent, aspect='auto', cmap='gist_heat_r')
cb = plt.colorbar()
cb.set_label('a (laser)')
plt.ylabel('r (microns)')

plt.subplot(312)
plt.imshow(2e18*n, extent=extent, aspect='auto', cmap='gist_heat_r', vmax=2.5e18)
cb = plt.colorbar()
cb.set_label('$n_e\;(cm^{-2})$')
plt.ylabel('r (microns)')

plt.subplot(313)
plt.imshow(T, extent=extent, aspect='auto', cmap='gist_heat_r', vmax=20)
cb = plt.colorbar()
cb.set_label('T (eV)')
plt.ylabel('r (microns)')

In [ ]:
#example of writing and reading gaussian laser intensity from file and plotting 1d temperature

lambd = 0.8e-6
tau = 50e-15 # FWHM duration
ell = np.array([1, 0])  # linear polarization

# Example: 1D radial intensity profile (cylindrical symmetry)
print("Example 1: 1D radial profile")
r_1d = np.linspace(0, 100e-6, 100)  # radial coordinate from 0 to 100 microns
w0_1d = 20e-6  # beam waist
I0_1d = 1e19  # peak intensity in W/m^2
intensity_1d = I0_1d * np.exp(-2*r_1d**2/w0_1d**2)

sample_1d_data = intensity_1d
np.savetxt('sample_radial_intensity.txt', sample_1d_data)
np.savetxt('sample_radial_intensity.csv', sample_1d_data, delimiter=',')

# Load from files
intensity_from_txt = load_intensity_profile('sample_radial_intensity.txt')
intensity_from_csv = load_intensity_profile('sample_radial_intensity.csv')

print(f"Loaded 1D profile shape: {intensity_from_txt.shape}")
print(f"Data match: {np.allclose(intensity_from_txt, intensity_from_csv)}")
pops = np.array([0.95, 0.0, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0])
# Process the loaded 1D data
ioniz_frac_file, T_eV_file = process_intensity_array_multispecies(intensity_from_txt, lambd, tau, ell,
    adk_prefactors, adk_powers, adk_exp_prefactors, source_indices, target_indices, charges, species_keys, pops,
    output_file='plasma_from_file_1d.h5', grid_extent={'r': [r_1d.min(), r_1d.max()]})

plt.figure(figsize=(6, 6))

plt.plot(r_1d*1e6, T_eV_file, 'b-', linewidth=2, label='Original 1D')
plt.xlabel('Radius (μm)')
plt.ylabel('Temperature (eV)')
plt.title('1D Temperature Profile\n(from file)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Clean up sample files
import os
os.remove('sample_radial_intensity.txt')
os.remove('sample_radial_intensity.csv')